In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import ParameterGrid


In [2]:
train_df = pd.read_csv("train.csv")
val_df = pd.read_csv("validation.csv")
test_df = pd.read_csv("test.csv")

print(f"Train: {train_df.shape}")
print(f"Val:   {val_df.shape}")
print(f"Test:  {test_df.shape}")


Train: (129477, 46)
Val:   (43755, 46)
Test:  (12853, 46)


In [3]:
TARGET = "log_price"
DROP_COLS = ["ClosePrice", "CloseDate", "price_ratio", TARGET]
feature_cols = [c for c in train_df.columns if c not in DROP_COLS]

X_train, y_train = train_df[feature_cols], train_df[TARGET]
X_val, y_val = val_df[feature_cols], val_df[TARGET]
X_test, y_test = test_df[feature_cols], test_df[TARGET]

RANDOM_STATE = 20260805

In [4]:
def evaluate(y_true_log, y_pred_log, label=""):
    y_true = np.exp(y_true_log)
    y_pred = np.exp(y_pred_log)

    rmse = root_mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    print(
        f"{label:>10} | RMSE: ${rmse:,.0f}  MAE: ${mae:,.0f}  R2: {r2:.4f}  MAPE: {mape:.2f}%"
    )
    return {"rmse": rmse, "mae": mae, "r2": r2, "mape": mape}


## Outlier

In [5]:
lower_cap = train_df["price_ratio"].quantile(0.005)
upper_cap = train_df["price_ratio"].quantile(0.9995)
print(f"Lower cap: {lower_cap:.4f}, Upper cap: {upper_cap:.4f}")

train_mask = (train_df["price_ratio"] >= lower_cap) & (
    train_df["price_ratio"] <= upper_cap
)
val_mask = (val_df["price_ratio"] >= lower_cap) & (val_df["price_ratio"] <= upper_cap)
test_mask = (test_df["price_ratio"] >= lower_cap) & (
    test_df["price_ratio"] <= upper_cap
)

print(
    f"Train: {train_mask.sum()} / {len(train_df)} kept ({(1 - train_mask.mean()) * 100:.4f}% removed)"
)
print(
    f"Val: {val_mask.sum()} / {len(val_df)} kept ({(1 - val_mask.mean()) * 100:.4f}% removed)"
)
print(
    f"Test: {test_mask.sum()} / {len(test_df)} kept ({(1 - test_mask.mean()) * 100:.4f}% removed)"
)


Lower cap: 0.7893, Upper cap: 1.9807
Train: 128764 / 129477 kept (0.5507% removed)
Val: 43507 / 43755 kept (0.5668% removed)
Test: 12794 / 12853 kept (0.4590% removed)


In [6]:
X_train_no_outliers = X_train[train_mask]
y_train_no_outliers = y_train[train_mask]

X_val_no_outliers = X_val[val_mask]
y_val_no_outliers = y_val[val_mask]

X_test_no_outliers = X_test[test_mask]
y_test_no_outliers = y_test[test_mask]


## XGBoost

## Baseline with and without outliers

In [7]:
xgb_base = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
xgb_base.fit(X_train, y_train)
evaluate(y_val, xgb_base.predict(X_val), "XGB base")


  XGB base | RMSE: $8,237,289  MAE: $284,916  R2: 0.0236  MAPE: 16.11%


{'rmse': 8237288.7267421335,
 'mae': 284916.1648652443,
 'r2': 0.023611924365872672,
 'mape': np.float64(16.105077621808135)}

In [8]:
xgb_base_no_outliers = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
xgb_base_no_outliers.fit(X_train_no_outliers, y_train_no_outliers)
evaluate(
    y_val_no_outliers,
    xgb_base_no_outliers.predict(X_val_no_outliers),
    "XGB base without outliers",
)


XGB base without outliers | RMSE: $578,809  MAE: $181,277  R2: 0.8373  MAPE: 11.95%


{'rmse': 578808.6798516219,
 'mae': 181277.08293508805,
 'r2': 0.8372785559099081,
 'mape': np.float64(11.953563885632816)}

### hyperparameter tuning

In [9]:
param_grid = {
    "max_depth": [4, 5, 6, 7],
    "learning_rate": [0.05, 0.1, 0.15],
    "n_estimators": [300, 600],
}


def grid_search_xgb(param_grid, X_train, y_train, X_val, y_val):
    results = []
    for params in ParameterGrid(param_grid):
        model = xgb.XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1, **params)
        model.fit(X_train, y_train)
        pred = model.predict(X_val)
        rmse = root_mean_squared_error(np.exp(y_val), np.exp(pred))
        results.append({**params, "val_rmse": rmse})
    return pd.DataFrame(results).sort_values("val_rmse").reset_index(drop=True)


xgb_results = grid_search_xgb(
    param_grid,
    X_train_no_outliers,
    y_train_no_outliers,
    X_val_no_outliers,
    y_val_no_outliers,
)
xgb_results


,learning_rate,max_depth,n_estimators,val_rmse
0,0.10,6,600,576634.411894
1,0.10,6,300,578808.679852
2,0.15,6,300,579791.223081
3,0.05,6,600,579876.978253
4,0.15,6,600,582739.090384
5,0.15,7,600,585625.691365
6,0.15,7,300,586173.247462
7,0.15,4,600,586847.874875
8,0.05,6,300,588416.213809
9,0.10,5,600,589578.490183


In [10]:
comparison_results = {}

comparison_results["Baseline (with outliers)"] = evaluate(
    y_val, xgb_base.predict(X_val), "XGB base w/ outliers"
)

comparison_results["Baseline (no outliers)"] = evaluate(
    y_val_no_outliers, xgb_base_no_outliers.predict(X_val_no_outliers), "XGB base clean"
)

best_params = xgb_results.iloc[0][
    ["max_depth", "learning_rate", "n_estimators"]
].to_dict()
best_params["max_depth"] = int(best_params["max_depth"])
best_params["n_estimators"] = int(best_params["n_estimators"])

xgb_tuned = xgb.XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1, **best_params)
xgb_tuned.fit(X_train_no_outliers, y_train_no_outliers)
comparison_results["Tuned (no outliers)"] = evaluate(
    y_val_no_outliers, xgb_tuned.predict(X_val_no_outliers), "XGB tuned"
)


XGB base w/ outliers | RMSE: $8,237,289  MAE: $284,916  R2: 0.0236  MAPE: 16.11%
XGB base clean | RMSE: $578,809  MAE: $181,277  R2: 0.8373  MAPE: 11.95%
 XGB tuned | RMSE: $576,634  MAE: $176,148  R2: 0.8385  MAPE: 11.57%


In [11]:
comparison_df = pd.DataFrame(comparison_results).T
comparison_df["rmse"] = comparison_df["rmse"].map(lambda x: f"${x:,.0f}")
comparison_df["mae"] = comparison_df["mae"].map(lambda x: f"${x:,.0f}")
comparison_df["r2"] = comparison_df["r2"].map(lambda x: f"{x:.4f}")
comparison_df["mape"] = comparison_df["mape"].map(lambda x: f"{x:.2f}%")
comparison_df


,rmse,mae,r2,mape
Baseline (with outliers),"$8,237,289","$284,916",0.0236,16.11%
Baseline (no outliers),"$578,809","$181,277",0.8373,11.95%
Tuned (no outliers),"$576,634","$176,148",0.8385,11.57%


## LightGBM

In [12]:
lgb_base = lgb.LGBMRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
lgb_base.fit(X_train, y_train)
evaluate(y_val, lgb_base.predict(X_val), "LGBM base")


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004632 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3477
[LightGBM] [Info] Number of data points in the train set: 129477, number of used features: 35
[LightGBM] [Info] Start training from score 13.777438
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

{'rmse': 8237850.519271379,
 'mae': 287350.32955997874,
 'r2': 0.023478738253345566,
 'mape': np.float64(16.248495671501605)}

In [13]:
lgb_base_no_outliers = lgb.LGBMRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
lgb_base_no_outliers.fit(X_train_no_outliers, y_train_no_outliers)
evaluate(
    y_val_no_outliers,
    lgb_base_no_outliers.predict(X_val_no_outliers),
    "LGBM base without outliers",
)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003626 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3475
[LightGBM] [Info] Number of data points in the train set: 128764, number of used features: 35
[LightGBM] [Info] Start training from score 13.779382
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

{'rmse': 578443.8767571585,
 'mae': 184798.65965209858,
 'r2': 0.837483606669243,
 'mape': np.float64(12.209129602053066)}

In [14]:
def grid_search_lgb(param_grid, X_train, y_train, X_val, y_val):
    results = []
    for params in ParameterGrid(param_grid):
        model = lgb.LGBMRegressor(random_state=RANDOM_STATE, n_jobs=-1, **params)
        model.fit(X_train, y_train)
        pred = model.predict(X_val)
        rmse = root_mean_squared_error(np.exp(y_val), np.exp(pred))
        results.append({**params, "val_rmse": rmse})
    return pd.DataFrame(results).sort_values("val_rmse").reset_index(drop=True)


param_grid = {
    "max_depth": [4, 5, 6, 7],
    "learning_rate": [0.05, 0.1, 0.15],
    "n_estimators": [300, 600],
}

lgb_results = grid_search_lgb(
    param_grid,
    X_train_no_outliers,
    y_train_no_outliers,
    X_val_no_outliers,
    y_val_no_outliers,
)
lgb_results


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004407 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3475
[LightGBM] [Info] Number of data points in the train set: 128764, number of used features: 35
[LightGBM] [Info] Start training from score 13.779382
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

,learning_rate,max_depth,n_estimators,val_rmse
0,0.15,7,600,563212.113016
1,0.15,7,300,565932.300437
2,0.15,6,300,568841.094821
3,0.10,6,600,569673.105413
4,0.15,5,600,572817.941663
5,0.15,5,300,572980.434061
6,0.15,4,600,574216.284533
7,0.15,6,600,574783.071269
8,0.05,7,600,577964.159461
9,0.10,7,600,578232.956126


In [15]:
comparison_results_lgb = {}

comparison_results_lgb["Baseline (with outliers)"] = evaluate(
    y_val, lgb_base.predict(X_val), "LGBM base w/ outliers"
)

comparison_results_lgb["Baseline (no outliers)"] = evaluate(
    y_val_no_outliers,
    lgb_base_no_outliers.predict(X_val_no_outliers),
    "LGBM base clean",
)

best_params_lgb = lgb_results.iloc[0][
    ["max_depth", "learning_rate", "n_estimators"]
].to_dict()
best_params_lgb["max_depth"] = int(best_params_lgb["max_depth"])
best_params_lgb["n_estimators"] = int(best_params_lgb["n_estimators"])

lgb_tuned = lgb.LGBMRegressor(random_state=RANDOM_STATE, n_jobs=-1, **best_params_lgb)
lgb_tuned.fit(X_train_no_outliers, y_train_no_outliers)
comparison_results_lgb["Tuned (no outliers)"] = evaluate(
    y_val_no_outliers, lgb_tuned.predict(X_val_no_outliers), "LGBM tuned"
)


LGBM base w/ outliers | RMSE: $8,237,851  MAE: $287,350  R2: 0.0235  MAPE: 16.25%
LGBM base clean | RMSE: $578,444  MAE: $184,799  R2: 0.8375  MAPE: 12.21%
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003733 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3475
[LightGBM] [Info] Number of data points in the train set: 128764, number of used features: 35
[LightGBM] [Info] Start training from score 13.779382
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

In [16]:
comparison_df_lgb = pd.DataFrame(comparison_results_lgb).T
comparison_df_lgb["rmse"] = comparison_df_lgb["rmse"].map(lambda x: f"${x:,.0f}")
comparison_df_lgb["mae"] = comparison_df_lgb["mae"].map(lambda x: f"${x:,.0f}")
comparison_df_lgb["r2"] = comparison_df_lgb["r2"].map(lambda x: f"{x:.4f}")
comparison_df_lgb["mape"] = comparison_df_lgb["mape"].map(lambda x: f"{x:.2f}%")
comparison_df_lgb


,rmse,mae,r2,mape
Baseline (with outliers),"$8,237,851","$287,350",0.0235,16.25%
Baseline (no outliers),"$578,444","$184,799",0.8375,12.21%
Tuned (no outliers),"$563,212","$176,711",0.8459,11.65%


In [17]:
final_comparison = {
    "XGBoost (tuned)": evaluate(
        y_val_no_outliers, xgb_tuned.predict(X_val_no_outliers), "XGB tuned"
    ),
    "LightGBM (tuned)": evaluate(
        y_val_no_outliers, lgb_tuned.predict(X_val_no_outliers), "LGBM tuned"
    ),
}

final_comparison_df = pd.DataFrame(final_comparison).T
final_comparison_df["rmse"] = final_comparison_df["rmse"].map(lambda x: f"${x:,.0f}")
final_comparison_df["mae"] = final_comparison_df["mae"].map(lambda x: f"${x:,.0f}")
final_comparison_df["r2"] = final_comparison_df["r2"].map(lambda x: f"{x:.4f}")
final_comparison_df["mape"] = final_comparison_df["mape"].map(lambda x: f"{x:.2f}%")
final_comparison_df


 XGB tuned | RMSE: $576,634  MAE: $176,148  R2: 0.8385  MAPE: 11.57%
LGBM tuned | RMSE: $563,212  MAE: $176,711  R2: 0.8459  MAPE: 11.65%


,rmse,mae,r2,mape
XGBoost (tuned),"$576,634","$176,148",0.8385,11.57%
LightGBM (tuned),"$563,212","$176,711",0.8459,11.65%


In [18]:
RANDOM_STATE = 20260805


def evaluate(y_true_log, y_pred_log, label=""):
    y_true = np.exp(y_true_log)
    y_pred = np.exp(y_pred_log)

    rmse = root_mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    print(
        f"{label:>10} | RMSE: ${rmse:,.0f}  MAE: ${mae:,.0f}  R2: {r2:.4f}  MAPE: {mape:.2f}%"
    )
    return {"rmse": rmse, "mae": mae, "r2": r2, "mape": mape}


from sklearn.tree import DecisionTreeRegressor

dt_check = DecisionTreeRegressor(max_depth=10, random_state=RANDOM_STATE)
dt_check.fit(X_train_no_outliers, y_train_no_outliers)
evaluate(y_val_no_outliers, dt_check.predict(X_val_no_outliers), "DT check")


  DT check | RMSE: $699,354  MAE: $236,600  R2: 0.7624  MAPE: 15.78%


{'rmse': 699354.3651372286,
 'mae': 236600.1564947272,
 'r2': 0.762442197475818,
 'mape': np.float64(15.779801717203569)}

In [19]:
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import make_scorer


def r2_on_price_scale(y_true_log, y_pred_log):
    return r2_score(np.exp(y_true_log), np.exp(y_pred_log))


price_scale_r2 = make_scorer(r2_on_price_scale, greater_is_better=True)
tscv = TimeSeriesSplit(n_splits=5)

param_grid_dt = {"max_depth": [3, 5, 7, 10, 15, 20, None]}

grid_search_dt = GridSearchCV(
    estimator=DecisionTreeRegressor(random_state=RANDOM_STATE),
    param_grid=param_grid_dt,
    cv=tscv,
    scoring=price_scale_r2,
    n_jobs=-1,
)
grid_search_dt.fit(X_train_no_outliers, y_train_no_outliers)
print(f"Best max_depth: {grid_search_dt.best_params_}")

dt_tuned = DecisionTreeRegressor(
    max_depth=grid_search_dt.best_params_["max_depth"],
    random_state=RANDOM_STATE,
)
dt_tuned.fit(X_train_no_outliers, y_train_no_outliers)
evaluate(
    y_val_no_outliers,
    dt_tuned.predict(X_val_no_outliers),
    "DT tuned",
)


Best max_depth: {'max_depth': 7}
  DT tuned | RMSE: $733,079  MAE: $261,653  R2: 0.7390  MAPE: 17.64%


{'rmse': 733079.2234448927,
 'mae': 261652.85078180063,
 'r2': 0.738978343395463,
 'mape': np.float64(17.64157681564917)}

In [20]:
from sklearn.ensemble import RandomForestRegressor

param_grid_rf = {
    "n_estimators": [100, 200, 300],
    "max_depth": [10, 15, 20, None],
    "max_features": ["sqrt", "log2", 0.5],
}

grid_search_rf = GridSearchCV(
    estimator=RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    param_grid=param_grid_rf,
    cv=tscv,
    scoring=price_scale_r2,
    n_jobs=1,
)
grid_search_rf.fit(X_train_no_outliers, y_train_no_outliers)
print(f"Best params: {grid_search_rf.best_params_}")

rf_tuned = RandomForestRegressor(
    **grid_search_rf.best_params_, random_state=RANDOM_STATE, n_jobs=-1
)
rf_tuned.fit(X_train_no_outliers, y_train_no_outliers)
evaluate(
    y_val_no_outliers,
    rf_tuned.predict(X_val_no_outliers),
    "RF tuned",
)


Best params: {'max_depth': None, 'max_features': 0.5, 'n_estimators': 300}
  RF tuned | RMSE: $597,341  MAE: $183,126  R2: 0.8267  MAPE: 11.94%


{'rmse': 597341.1821630792,
 'mae': 183126.12780495503,
 'r2': 0.8266915917148113,
 'mape': np.float64(11.943232260600741)}

In [21]:
final_train_val = {
    "Decision Tree (tuned)": {
        "train_r2": r2_score(
            np.exp(y_train_no_outliers), np.exp(dt_tuned.predict(X_train_no_outliers))
        ),
        "val_r2": r2_score(
            np.exp(y_val_no_outliers), np.exp(dt_tuned.predict(X_val_no_outliers))
        ),
    },
    "Random Forest (tuned)": {
        "train_r2": r2_score(
            np.exp(y_train_no_outliers), np.exp(rf_tuned.predict(X_train_no_outliers))
        ),
        "val_r2": r2_score(
            np.exp(y_val_no_outliers), np.exp(rf_tuned.predict(X_val_no_outliers))
        ),
    },
    "XGBoost (tuned)": {
        "train_r2": r2_score(
            np.exp(y_train_no_outliers), np.exp(xgb_tuned.predict(X_train_no_outliers))
        ),
        "val_r2": r2_score(
            np.exp(y_val_no_outliers), np.exp(xgb_tuned.predict(X_val_no_outliers))
        ),
    },
    "LightGBM (tuned)": {
        "train_r2": r2_score(
            np.exp(y_train_no_outliers), np.exp(lgb_tuned.predict(X_train_no_outliers))
        ),
        "val_r2": r2_score(
            np.exp(y_val_no_outliers), np.exp(lgb_tuned.predict(X_val_no_outliers))
        ),
    },
}

train_val_df = pd.DataFrame(final_train_val).T
train_val_df["gap"] = train_val_df["train_r2"] - train_val_df["val_r2"]
train_val_df = train_val_df.round(4)
train_val_df


,train_r2,val_r2,gap
Decision Tree (tuned),0.7442,0.7390,0.0053
Random Forest (tuned),0.9558,0.8267,0.1291
XGBoost (tuned),0.9547,0.8385,0.1162
LightGBM (tuned),0.9390,0.8459,0.0931


## Model Improvement Experiments

### 1. Weighted Training (Luxury Segment)


In [46]:
# Weighted training experiment: give higher weight to luxury properties (>$2M)
# to address the sparse-sample problem observed in the price band analysis

train_price = np.exp(y_train_no_outliers)
sample_weight = np.where(train_price > 2_000_000, 3, 1)

xgb_weighted = xgb.XGBRegressor(
    max_depth=6,
    learning_rate=0.1,
    n_estimators=600,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
xgb_weighted.fit(X_train_no_outliers, y_train_no_outliers, sample_weight=sample_weight)

print("=== Weighted vs. Unweighted (Validation Set) ===")
evaluate(y_val_no_outliers, xgb_tuned.predict(X_val_no_outliers), "XGB (no weight)")
evaluate(y_val_no_outliers, xgb_weighted.predict(X_val_no_outliers), "XGB (weighted)")

=== Weighted vs. Unweighted (Validation Set) ===
XGB (no weight) | RMSE: $576,634  MAE: $176,148  R2: 0.8385  MAPE: 11.57%
XGB (weighted) | RMSE: $578,776  MAE: $177,047  R2: 0.8373  MAPE: 11.84%


{'rmse': 578776.3110979927,
 'mae': 177047.39930452642,
 'r2': 0.8372967551637941,
 'mape': np.float64(11.84123817394761)}

In [48]:
def price_band_analysis(model, X_test, y_test_log, model_name=""):
    y_true = np.exp(y_test_log)
    y_pred = np.exp(model.predict(X_test))

    bands = pd.cut(
        y_true,
        bins=[0, 500_000, 1_000_000, 2_000_000, np.inf],
        labels=["<$500K", "$500K-$1M", "$1M-$2M", "$2M+"],
    )

    results = []
    for band in bands.cat.categories:
        mask = bands == band
        if mask.sum() == 0:
            continue
        yt, yp = y_true[mask], y_pred[mask]
        rmse = root_mean_squared_error(yt, yp)
        mae = mean_absolute_error(yt, yp)
        mape = np.mean(np.abs((yt - yp) / yt)) * 100
        mdape = np.median(np.abs((yt - yp) / yt)) * 100
        results.append(
            {
                "price_band": band,
                "n_listings": mask.sum(),
                "rmse": rmse,
                "mae": mae,
                "mape": mape,
                "mdape": mdape,
            }
        )

    band_df = pd.DataFrame(results)
    print(f"=== {model_name} — Performance by Price Band ===")
    print(band_df.to_string(index=False))
    return band_df


xgb_base_band_val = price_band_analysis(
    xgb_tuned, X_val_no_outliers, y_val_no_outliers, "XGB No Weight (Val)"
)

xgb_weighted_band_val = price_band_analysis(
    xgb_weighted, X_val_no_outliers, y_val_no_outliers, "XGB Weighted (Val)"
)


=== XGB No Weight (Val) — Performance by Price Band ===
price_band  n_listings         rmse           mae      mape     mdape
    <$500K        6212 8.080870e+04  50156.118379 15.161452  7.897761
 $500K-$1M       18254 9.998950e+04  67124.200161  9.001379  6.259503
   $1M-$2M       13006 2.296577e+05 162867.682696 11.515532  8.984236
      $2M+        6035 1.498818e+06 664215.285914 15.770309 12.561173
=== XGB Weighted (Val) — Performance by Price Band ===
price_band  n_listings         rmse           mae      mape     mdape
    <$500K        6212 8.161380e+04  50310.993736 15.164435  8.001587
 $500K-$1M       18254 1.042217e+05  68584.582342  9.184926  6.417561
   $1M-$2M       13006 2.512203e+05 177973.401456 12.579867  9.635050
      $2M+        6035 1.496396e+06 633571.546488 14.863278 11.445413


In [49]:
lgb_weighted = lgb.LGBMRegressor(
    max_depth=7,
    learning_rate=0.15,
    n_estimators=600,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
lgb_weighted.fit(X_train_no_outliers, y_train_no_outliers, sample_weight=sample_weight)

print("=== Weighted vs. Unweighted (Validation Set) ===")
lgb_base_val_pred = lgb_tuned.predict(X_val_no_outliers)
lgb_base_metrics = evaluate(y_val_no_outliers, lgb_base_val_pred, "LGBM (no weight)")

lgb_weighted_val_pred = lgb_weighted.predict(X_val_no_outliers)
lgb_weighted_metrics = evaluate(
    y_val_no_outliers, lgb_weighted_val_pred, "LGBM (weighted)"
)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004789 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3475
[LightGBM] [Info] Number of data points in the train set: 128764, number of used features: 35
[LightGBM] [Info] Start training from score 14.023010
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

In [51]:
lgb_base_band_val = price_band_analysis(
    lgb_tuned, X_val_no_outliers, y_val_no_outliers, "LGBM No Weight (Val)"
)
lgb_weighted_band_val = price_band_analysis(
    lgb_weighted, X_val_no_outliers, y_val_no_outliers, "LGBM Weighted (Val)"
)


=== LGBM No Weight (Val) — Performance by Price Band ===
price_band  n_listings         rmse           mae      mape     mdape
    <$500K        6212 8.085195e+04  50775.745331 15.342735  8.174854
 $500K-$1M       18254 1.006662e+05  67352.536773  9.027073  6.374573
   $1M-$2M       13006 2.299381e+05 163624.562530 11.560530  9.047927
      $2M+        6035 1.461323e+06 665316.608254 15.967270 12.884288
=== LGBM Weighted (Val) — Performance by Price Band ===
price_band  n_listings         rmse           mae      mape     mdape
    <$500K        6212 8.294768e+04  51002.076303 15.429168  8.067369
 $500K-$1M       18254 1.058441e+05  69047.232170  9.253058  6.475343
   $1M-$2M       13006 2.526308e+05 177669.154210 12.561777  9.693679
      $2M+        6035 1.496576e+06 633375.604092 14.847448 11.279915


### Ensemble (XGB + LGBM) 

In [52]:
xgb_w_pred_price = np.exp(xgb_weighted.predict(X_val_no_outliers))
lgb_w_pred_price = np.exp(lgb_weighted.predict(X_val_no_outliers))
y_true_price = np.exp(y_val_no_outliers)

ensemble_pred_price = 0.5 * xgb_w_pred_price + 0.5 * lgb_w_pred_price

rmse = root_mean_squared_error(y_true_price, ensemble_pred_price)
mae = mean_absolute_error(y_true_price, ensemble_pred_price)
r2 = r2_score(y_true_price, ensemble_pred_price)
mape = np.mean(np.abs((y_true_price - ensemble_pred_price) / y_true_price)) * 100
mdape = np.median(np.abs((y_true_price - ensemble_pred_price) / y_true_price)) * 100

print(
    f"Ensemble (weighted, 50/50, Val) | RMSE: ${rmse:,.0f}  MAE: ${mae:,.0f}  R2: {r2:.4f}  MAPE: {mape:.2f}%  MdAPE: {mdape:.2f}%"
)


Ensemble (weighted, 50/50, Val) | RMSE: $570,475  MAE: $174,936  R2: 0.8419  MAPE: 11.73%  MdAPE: 7.97%


In [26]:
best_r2 = -np.inf
best_w = None
for w in np.arange(0, 1.05, 0.1):
    pred = w * xgb_w_pred_price + (1 - w) * lgb_w_pred_price
    r2_w = r2_score(y_true_price, pred)
    print(f"XGB weight={w:.1f} | R²={r2_w:.4f}")
    if r2_w > best_r2:
        best_r2 = r2_w
        best_w = w

print(f"\nBest XGB weight: {best_w:.1f}, R²: {best_r2:.4f}")


XGB weight=0.0 | R²=0.8573
XGB weight=0.1 | R²=0.8580
XGB weight=0.2 | R²=0.8584
XGB weight=0.3 | R²=0.8585
XGB weight=0.4 | R²=0.8582
XGB weight=0.5 | R²=0.8575
XGB weight=0.6 | R²=0.8566
XGB weight=0.7 | R²=0.8553
XGB weight=0.8 | R²=0.8536
XGB weight=0.9 | R²=0.8517
XGB weight=1.0 | R²=0.8493

Best XGB weight: 0.3, R²: 0.8585


In [27]:
def price_band_analysis_ensemble(pred_price, y_true_log, model_name=""):
    y_true = np.exp(y_true_log)

    bands = pd.cut(
        y_true,
        bins=[0, 500_000, 1_000_000, 2_000_000, np.inf],
        labels=["<$500K", "$500K-$1M", "$1M-$2M", "$2M+"],
    )

    results = []
    for band in bands.cat.categories:
        mask = bands == band
        if mask.sum() == 0:
            continue
        yt, yp = y_true[mask], pred_price[mask]
        rmse = root_mean_squared_error(yt, yp)
        mae = mean_absolute_error(yt, yp)
        mape = np.mean(np.abs((yt - yp) / yt)) * 100
        mdape = np.median(np.abs((yt - yp) / yt)) * 100
        results.append(
            {
                "price_band": band,
                "n_listings": mask.sum(),
                "rmse": rmse,
                "mae": mae,
                "mape": mape,
                "mdape": mdape,
            }
        )

    band_df = pd.DataFrame(results)
    print(f"=== {model_name} — Performance by Price Band ===")
    print(band_df.to_string(index=False))
    return band_df


final_ensemble_pred = best_w * xgb_w_pred_price + (1 - best_w) * lgb_w_pred_price
ensemble_band_results = price_band_analysis_ensemble(
    final_ensemble_pred, y_test_no_outliers, "Ensemble (best weight)"
)


=== Ensemble (best weight) — Performance by Price Band ===
price_band  n_listings         rmse           mae      mape     mdape
    <$500K        1785 7.839699e+04  51247.781797 15.725567  8.614826
 $500K-$1M        5316 1.025071e+05  67950.420419  9.025757  6.296832
   $1M-$2M        3908 2.469308e+05 175914.860441 12.378271  9.564645
      $2M+        1785 1.412712e+06 645936.978303 14.788969 11.304604


### Regularization

In [28]:
# v1: baseline weighted model (for comparison)
train_r2 = r2_score(
    np.exp(y_train_no_outliers), np.exp(xgb_weighted.predict(X_train_no_outliers))
)
val_r2 = r2_score(
    np.exp(y_val_no_outliers), np.exp(xgb_weighted.predict(X_val_no_outliers))
)
print(
    f"Baseline (weighted, no regularization) | Train R²: {train_r2:.4f}, Val R²: {val_r2:.4f}, Gap: {train_r2 - val_r2:.4f}"
)


Baseline (weighted, no regularization) | Train R²: 0.9684, Val R²: 0.8373, Gap: 0.1311


In [29]:
# v2: add subsample + colsample_bytree
xgb_reg_v1 = xgb.XGBRegressor(
    max_depth=6,
    learning_rate=0.1,
    n_estimators=600,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
xgb_reg_v1.fit(X_train_no_outliers, y_train_no_outliers, sample_weight=sample_weight)

train_r2 = r2_score(
    np.exp(y_train_no_outliers), np.exp(xgb_reg_v1.predict(X_train_no_outliers))
)
val_r2 = r2_score(
    np.exp(y_val_no_outliers), np.exp(xgb_reg_v1.predict(X_val_no_outliers))
)
print(
    f"v1 (subsample + colsample_bytree) | Train R²: {train_r2:.4f}, Val R²: {val_r2:.4f}, Gap: {train_r2 - val_r2:.4f}"
)


v1 (subsample + colsample_bytree) | Train R²: 0.9704, Val R²: 0.8396, Gap: 0.1308


In [30]:
# v3: add min_child_weight
xgb_reg_v2 = xgb.XGBRegressor(
    max_depth=6,
    learning_rate=0.1,
    n_estimators=600,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
xgb_reg_v2.fit(X_train_no_outliers, y_train_no_outliers, sample_weight=sample_weight)

train_r2 = r2_score(
    np.exp(y_train_no_outliers), np.exp(xgb_reg_v2.predict(X_train_no_outliers))
)
val_r2 = r2_score(
    np.exp(y_val_no_outliers), np.exp(xgb_reg_v2.predict(X_val_no_outliers))
)
print(
    f"v2 (+min_child_weight) | Train R²: {train_r2:.4f}, Val R²: {val_r2:.4f}, Gap: {train_r2 - val_r2:.4f}"
)


v2 (+min_child_weight) | Train R²: 0.9684, Val R²: 0.8436, Gap: 0.1248


In [31]:
# v4: add L1/L2 regularization
xgb_reg_v3 = xgb.XGBRegressor(
    max_depth=6,
    learning_rate=0.1,
    n_estimators=600,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    reg_alpha=0.5,
    reg_lambda=1,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
xgb_reg_v3.fit(X_train_no_outliers, y_train_no_outliers, sample_weight=sample_weight)

train_r2 = r2_score(
    np.exp(y_train_no_outliers), np.exp(xgb_reg_v3.predict(X_train_no_outliers))
)
val_r2 = r2_score(
    np.exp(y_val_no_outliers), np.exp(xgb_reg_v3.predict(X_val_no_outliers))
)
print(
    f"v3 (+L1/L2) | Train R²: {train_r2:.4f}, Val R²: {val_r2:.4f}, Gap: {train_r2 - val_r2:.4f}"
)


v3 (+L1/L2) | Train R²: 0.9675, Val R²: 0.8498, Gap: 0.1177


In [32]:
lgb_reg_v3 = lgb.LGBMRegressor(
    max_depth=7,
    learning_rate=0.15,
    n_estimators=600,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=20,  # LightGBM 用 min_child_samples 取代 min_child_weight
    reg_alpha=0.5,
    reg_lambda=1,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
lgb_reg_v3.fit(X_train_no_outliers, y_train_no_outliers, sample_weight=sample_weight)

train_r2 = r2_score(
    np.exp(y_train_no_outliers), np.exp(lgb_reg_v3.predict(X_train_no_outliers))
)
val_r2 = r2_score(
    np.exp(y_val_no_outliers), np.exp(lgb_reg_v3.predict(X_val_no_outliers))
)
print(
    f"LGBM v3 (regularized) | Train R²: {train_r2:.4f}, Val R²: {val_r2:.4f}, Gap: {train_r2 - val_r2:.4f}"
)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003948 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3475
[LightGBM] [Info] Number of data points in the train set: 128764, number of used features: 35
[LightGBM] [Info] Start training from score 14.023010
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
LGBM v3 (regularized) | Train R²: 0.9539, Val R²: 0.8445, Gap: 0.10

In [33]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "max_depth": [3, 4, 5, 6, 7],
    "learning_rate": [0.03, 0.05, 0.08, 0.1, 0.15],
    "n_estimators": [200, 300, 400, 600],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5, 10],
    "reg_alpha": [0, 0.1, 0.5, 1, 2],
    "reg_lambda": [1, 1.5, 2, 3],
}

random_search_xgb = RandomizedSearchCV(
    estimator=xgb.XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=50,
    cv=tscv,
    scoring=price_scale_r2,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)

random_search_xgb.fit(
    X_train_no_outliers,
    y_train_no_outliers,
    sample_weight=sample_weight,
)

print(f"Best params: {random_search_xgb.best_params_}")
print(f"Best CV R²: {random_search_xgb.best_score_:.4f}")


Fitting 5 folds for each of 50 candidates, totalling 250 fits


/opt/base-uv/.venv/lib/python3.13/site-packages/sklearn/model_selection/_search.py:883: UserWarning: The scoring make_scorer(r2_on_price_scale, response_method='predict') does not support sample_weight, which may lead to statistically incorrect results when fitting RandomizedSearchCV(cv=TimeSeriesSplit(gap=0, max_train_size=None, n_splits=5, test_size=None),
                   estimator=XGBRegressor(base_score=None, booster=None,
                                          callbacks=None,
                                          colsample_bylevel=None,
                                          colsample_bynode=None,
                                          colsample_bytree=None, device=None,
                                          early_stopping_rounds=None,
                                          enable_categorical=False,
                                          eval_metric=None, feature_types=None,
                                          feature_weights=None, gamm...
         

Best params: {'subsample': 0.8, 'reg_lambda': 3, 'reg_alpha': 2, 'n_estimators': 600, 'min_child_weight': 10, 'max_depth': 4, 'learning_rate': 0.08, 'colsample_bytree': 0.7}
Best CV R²: 0.8178


In [34]:
xgb_auto = xgb.XGBRegressor(
    subsample=0.8,
    reg_lambda=3,
    reg_alpha=2,
    n_estimators=600,
    min_child_weight=10,
    max_depth=4,
    learning_rate=0.08,
    colsample_bytree=0.7,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
xgb_auto.fit(X_train_no_outliers, y_train_no_outliers, sample_weight=sample_weight)

train_r2 = r2_score(
    np.exp(y_train_no_outliers), np.exp(xgb_auto.predict(X_train_no_outliers))
)
val_r2 = r2_score(
    np.exp(y_val_no_outliers), np.exp(xgb_auto.predict(X_val_no_outliers))
)
print(
    f"Auto-tuned XGB | Train R²: {train_r2:.4f}, Val R²: {val_r2:.4f}, Gap: {train_r2 - val_r2:.4f}"
)


Auto-tuned XGB | Train R²: 0.8971, Val R²: 0.8417, Gap: 0.0554


In [35]:
param_dist_lgb = {
    "max_depth": [3, 4, 5, 6, 7],
    "learning_rate": [0.03, 0.05, 0.08, 0.1, 0.15],
    "n_estimators": [200, 300, 400, 600],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "min_child_samples": [5, 10, 20, 30],
    "reg_alpha": [0, 0.1, 0.5, 1, 2],
    "reg_lambda": [1, 1.5, 2, 3],
}

random_search_lgb = RandomizedSearchCV(
    estimator=lgb.LGBMRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions=param_dist_lgb,
    n_iter=50,
    cv=tscv,
    scoring=price_scale_r2,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)

random_search_lgb.fit(
    X_train_no_outliers,
    y_train_no_outliers,
    sample_weight=sample_weight,
)

print(f"Best params: {random_search_lgb.best_params_}")
print(f"Best CV R²: {random_search_lgb.best_score_:.4f}")


Fitting 5 folds for each of 50 candidates, totalling 250 fits


/opt/base-uv/.venv/lib/python3.13/site-packages/sklearn/model_selection/_search.py:883: UserWarning: The scoring make_scorer(r2_on_price_scale, response_method='predict') does not support sample_weight, which may lead to statistically incorrect results when fitting RandomizedSearchCV(cv=TimeSeriesSplit(gap=0, max_train_size=None, n_splits=5, test_size=None),
                   estimator=LGBMRegressor(n_jobs=-1, random_state=20260805),
                   n_iter=50, n_jobs=-1,
                   param_distributions={'colsample_bytree': [0.6, 0.7, 0.8, 0.9,
                                                             1.0],
                                        'learning_rate': [0.03, 0.05, 0.08, 0.1,
                                                          0.15],
                                        'max_depth': [3, 4, 5, 6, 7],
                                        'min_child_samples': [5, 10, 20, 30],
                                        'n_estimators': [200, 300, 400, 600],


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003620 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3475
[LightGBM] [Info] Number of data points in the train set: 128764, number of used features: 35
[LightGBM] [Info] Start training from score 14.023010
Best params: {'subsample': 0.7, 'reg_lambda': 1, 'reg_alpha': 2, 'n_estimators': 300, 'min_child_samples': 10, 'max_depth': 7, 'learning_rate': 0.1, 'colsample_bytree': 0.7}
Best CV R²: 0.8126


In [36]:
lgb_auto = lgb.LGBMRegressor(
    subsample=0.7,
    reg_lambda=1,
    reg_alpha=2,
    n_estimators=300,
    min_child_samples=10,
    max_depth=7,
    learning_rate=0.1,
    colsample_bytree=0.7,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
lgb_auto.fit(X_train_no_outliers, y_train_no_outliers, sample_weight=sample_weight)

train_r2 = r2_score(
    np.exp(y_train_no_outliers), np.exp(lgb_auto.predict(X_train_no_outliers))
)
val_r2 = r2_score(
    np.exp(y_val_no_outliers), np.exp(lgb_auto.predict(X_val_no_outliers))
)
print(
    f"Auto-tuned LGBM | Train R²: {train_r2:.4f}, Val R²: {val_r2:.4f}, Gap: {train_r2 - val_r2:.4f}"
)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003432 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3475
[LightGBM] [Info] Number of data points in the train set: 128764, number of used features: 35
[LightGBM] [Info] Start training from score 14.023010
Auto-tuned LGBM | Train R²: 0.9125, Val R²: 0.8386, Gap: 0.0739


In [37]:
xgb_auto_val_pred = np.exp(xgb_auto.predict(X_val_no_outliers))
lgb_auto_val_pred = np.exp(lgb_auto.predict(X_val_no_outliers))
y_val_price = np.exp(y_val_no_outliers)

best_r2 = -np.inf
best_w = None
for w in np.arange(0, 1.05, 0.1):
    pred = w * xgb_auto_val_pred + (1 - w) * lgb_auto_val_pred
    r2_w = r2_score(y_val_price, pred)
    print(f"XGB weight={w:.1f} | R²={r2_w:.4f}")
    if r2_w > best_r2:
        best_r2 = r2_w
        best_w = w

print(f"\nBest XGB weight: {best_w:.1f}, R²: {best_r2:.4f}")


XGB weight=0.0 | R²=0.8386
XGB weight=0.1 | R²=0.8397
XGB weight=0.2 | R²=0.8407
XGB weight=0.3 | R²=0.8415
XGB weight=0.4 | R²=0.8420
XGB weight=0.5 | R²=0.8425
XGB weight=0.6 | R²=0.8427
XGB weight=0.7 | R²=0.8427
XGB weight=0.8 | R²=0.8426
XGB weight=0.9 | R²=0.8422
XGB weight=1.0 | R²=0.8417

Best XGB weight: 0.7, R²: 0.8427


In [38]:
xgb_auto_train_pred = np.exp(xgb_auto.predict(X_train_no_outliers))
lgb_auto_train_pred = np.exp(lgb_auto.predict(X_train_no_outliers))
y_train_price = np.exp(y_train_no_outliers)

ensemble_train_pred = best_w * xgb_auto_train_pred + (1 - best_w) * lgb_auto_train_pred
ensemble_val_pred = best_w * xgb_auto_val_pred + (1 - best_w) * lgb_auto_val_pred

train_r2_new = r2_score(y_train_price, ensemble_train_pred)
val_r2_new = r2_score(y_val_price, ensemble_val_pred)

print(
    f"New Ensemble (auto-tuned) | Train R²: {train_r2_new:.4f}, Val R²: {val_r2_new:.4f}, Gap: {train_r2_new - val_r2_new:.4f}"
)


New Ensemble (auto-tuned) | Train R²: 0.9035, Val R²: 0.8427, Gap: 0.0608


## New Ensemble (Auto-Tuned)

In [39]:
xgb_auto_test_pred = np.exp(xgb_auto.predict(X_test_no_outliers))
lgb_auto_test_pred = np.exp(lgb_auto.predict(X_test_no_outliers))
y_test_price = np.exp(y_test_no_outliers)

final_ensemble_test_pred = (
    best_w * xgb_auto_test_pred + (1 - best_w) * lgb_auto_test_pred
)

rmse = root_mean_squared_error(y_test_price, final_ensemble_test_pred)
mae = mean_absolute_error(y_test_price, final_ensemble_test_pred)
r2 = r2_score(y_test_price, final_ensemble_test_pred)
mape = np.mean(np.abs((y_test_price - final_ensemble_test_pred) / y_test_price)) * 100
mdape = (
    np.median(np.abs((y_test_price - final_ensemble_test_pred) / y_test_price)) * 100
)

print(
    f"Final Ensemble (Test Set) | RMSE: ${rmse:,.0f}  MAE: ${mae:,.0f}  R2: {r2:.4f}  MAPE: {mape:.2f}%  MdAPE: {mdape:.2f}%"
)


Final Ensemble (Test Set) | RMSE: $565,695  MAE: $190,045  R2: 0.8502  MAPE: 12.71%  MdAPE: 8.67%


In [40]:
def price_band_analysis_ensemble(pred_price, y_true_log, model_name=""):
    y_true = np.exp(y_true_log)

    bands = pd.cut(
        y_true,
        bins=[0, 500_000, 1_000_000, 2_000_000, np.inf],
        labels=["<$500K", "$500K-$1M", "$1M-$2M", "$2M+"],
    )

    results = []
    for band in bands.cat.categories:
        mask = bands == band
        if mask.sum() == 0:
            continue
        yt, yp = y_true[mask], pred_price[mask]
        rmse = root_mean_squared_error(yt, yp)
        mae = mean_absolute_error(yt, yp)
        mape = np.mean(np.abs((yt - yp) / yt)) * 100
        mdape = np.median(np.abs((yt - yp) / yt)) * 100
        results.append(
            {
                "price_band": band,
                "n_listings": mask.sum(),
                "rmse": rmse,
                "mae": mae,
                "mape": mape,
                "mdape": mdape,
            }
        )

    band_df = pd.DataFrame(results)
    print(f"=== {model_name} — Performance by Price Band ===")
    print(band_df.to_string(index=False))
    return band_df


final_band_results = price_band_analysis_ensemble(
    final_ensemble_test_pred, y_test_no_outliers, "Final Ensemble (Auto-Tuned)"
)


=== Final Ensemble (Auto-Tuned) — Performance by Price Band ===
price_band  n_listings         rmse           mae      mape     mdape
    <$500K        1785 8.249299e+04  54781.161878 16.938827  9.175216
 $500K-$1M        5316 1.121193e+05  74083.629180  9.841368  6.836503
   $1M-$2M        3908 2.622355e+05 190442.385709 13.437987 10.331895
      $2M+        1785 1.448753e+06 669792.904528 15.455625 12.209828


In [53]:
final_summary_rows = []


def get_r2(model, X, y_log):
    return r2_score(np.exp(y_log), np.exp(model.predict(X)))


# 1. XGB / LGBM baseline (tuned, no weighting, no reg)
xgb_base_mape, xgb_base_mdape = dollar_metrics(
    y_test_no_outliers, xgb_tuned.predict(X_test_no_outliers)
)
lgb_base_mape, lgb_base_mdape = dollar_metrics(
    y_test_no_outliers, lgb_tuned.predict(X_test_no_outliers)
)

final_summary_rows.append(
    {
        "stage": "XGB baseline (tuned)",
        "train_r2": get_r2(xgb_tuned, X_train_no_outliers, y_train_no_outliers),
        "val_r2": get_r2(xgb_tuned, X_val_no_outliers, y_val_no_outliers),
        "test_r2": get_r2(xgb_tuned, X_test_no_outliers, y_test_no_outliers),
        "mape": xgb_base_mape,
        "mdape": xgb_base_mdape,
    }
)
final_summary_rows.append(
    {
        "stage": "LGBM baseline (tuned)",
        "train_r2": get_r2(lgb_tuned, X_train_no_outliers, y_train_no_outliers),
        "val_r2": get_r2(lgb_tuned, X_val_no_outliers, y_val_no_outliers),
        "test_r2": get_r2(lgb_tuned, X_test_no_outliers, y_test_no_outliers),
        "mape": lgb_base_mape,
        "mdape": lgb_base_mdape,
    }
)

# 2. Weighted (no reg)
xgb_w_mape, xgb_w_mdape = dollar_metrics(
    y_test_no_outliers, xgb_weighted.predict(X_test_no_outliers)
)
lgb_w_mape, lgb_w_mdape = dollar_metrics(
    y_test_no_outliers, lgb_weighted.predict(X_test_no_outliers)
)

final_summary_rows.append(
    {
        "stage": "XGB weighted (no reg)",
        "train_r2": get_r2(xgb_weighted, X_train_no_outliers, y_train_no_outliers),
        "val_r2": get_r2(xgb_weighted, X_val_no_outliers, y_val_no_outliers),
        "test_r2": get_r2(xgb_weighted, X_test_no_outliers, y_test_no_outliers),
        "mape": xgb_w_mape,
        "mdape": xgb_w_mdape,
    }
)
final_summary_rows.append(
    {
        "stage": "LGBM weighted (no reg)",
        "train_r2": get_r2(lgb_weighted, X_train_no_outliers, y_train_no_outliers),
        "val_r2": get_r2(lgb_weighted, X_val_no_outliers, y_val_no_outliers),
        "test_r2": get_r2(lgb_weighted, X_test_no_outliers, y_test_no_outliers),
        "mape": lgb_w_mape,
        "mdape": lgb_w_mdape,
    }
)

# 3. Ensemble (weighted, no reg) — best_w=0.3 from val search
xgb_w_train_pred = np.exp(xgb_weighted.predict(X_train_no_outliers))
lgb_w_train_pred = np.exp(lgb_weighted.predict(X_train_no_outliers))
xgb_w_val_pred_ = np.exp(xgb_weighted.predict(X_val_no_outliers))
lgb_w_val_pred_ = np.exp(lgb_weighted.predict(X_val_no_outliers))
xgb_w_test_pred = np.exp(xgb_weighted.predict(X_test_no_outliers))
lgb_w_test_pred = np.exp(lgb_weighted.predict(X_test_no_outliers))

y_train_price_ = np.exp(y_train_no_outliers)
y_val_price_ = np.exp(y_val_no_outliers)
y_test_price_ = np.exp(y_test_no_outliers)

ens_train_pred = 0.3 * xgb_w_train_pred + 0.7 * lgb_w_train_pred
ens_val_pred = 0.3 * xgb_w_val_pred_ + 0.7 * lgb_w_val_pred_
ens_test_pred = 0.3 * xgb_w_test_pred + 0.7 * lgb_w_test_pred

ens_mape = np.mean(np.abs((y_test_price_ - ens_test_pred) / y_test_price_)) * 100
ens_mdape = np.median(np.abs((y_test_price_ - ens_test_pred) / y_test_price_)) * 100

final_summary_rows.append(
    {
        "stage": "Ensemble (weighted, no reg)",
        "train_r2": r2_score(y_train_price_, ens_train_pred),
        "val_r2": r2_score(y_val_price_, ens_val_pred),
        "test_r2": r2_score(y_test_price_, ens_test_pred),
        "mape": ens_mape,
        "mdape": ens_mdape,
    }
)

# 4. Auto-tuned (weighted + regularized)
xgb_auto_mape, xgb_auto_mdape = dollar_metrics(
    y_test_no_outliers, xgb_auto.predict(X_test_no_outliers)
)
lgb_auto_mape, lgb_auto_mdape = dollar_metrics(
    y_test_no_outliers, lgb_auto.predict(X_test_no_outliers)
)

final_summary_rows.append(
    {
        "stage": "XGB auto-tuned (weighted + regularized)",
        "train_r2": get_r2(xgb_auto, X_train_no_outliers, y_train_no_outliers),
        "val_r2": get_r2(xgb_auto, X_val_no_outliers, y_val_no_outliers),
        "test_r2": get_r2(xgb_auto, X_test_no_outliers, y_test_no_outliers),
        "mape": xgb_auto_mape,
        "mdape": xgb_auto_mdape,
    }
)
final_summary_rows.append(
    {
        "stage": "LGBM auto-tuned (weighted + regularized)",
        "train_r2": get_r2(lgb_auto, X_train_no_outliers, y_train_no_outliers),
        "val_r2": get_r2(lgb_auto, X_val_no_outliers, y_val_no_outliers),
        "test_r2": get_r2(lgb_auto, X_test_no_outliers, y_test_no_outliers),
        "mape": lgb_auto_mape,
        "mdape": lgb_auto_mdape,
    }
)

# 5. Final Ensemble (auto-tuned)
final_ens_mape = (
    np.mean(np.abs((y_test_price - final_ensemble_test_pred) / y_test_price)) * 100
)
final_ens_mdape = (
    np.median(np.abs((y_test_price - final_ensemble_test_pred) / y_test_price)) * 100
)

final_summary_rows.append(
    {
        "stage": "Final Ensemble (auto-tuned)",
        "train_r2": train_r2_new,
        "val_r2": val_r2_new,
        "test_r2": r2_score(y_test_price, final_ensemble_test_pred),
        "mape": final_ens_mape,
        "mdape": final_ens_mdape,
    }
)

final_summary_table = pd.DataFrame(final_summary_rows)
for col in ["train_r2", "val_r2", "test_r2", "mape", "mdape"]:
    final_summary_table[col] = final_summary_table[col].round(4)
final_summary_table["gap"] = (
    final_summary_table["train_r2"] - final_summary_table["val_r2"]
).round(4)
final_summary_table


,stage,train_r2,val_r2,test_r2,mape,mdape,gap
0,XGB baseline (tuned),0.9547,0.8385,0.8473,11.5793,7.8786,0.1162
1,LGBM baseline (tuned),0.9390,0.8459,0.8480,11.6472,8.0228,0.0931
2,XGB weighted (no reg),0.9684,0.8373,0.8493,11.9094,8.1337,0.1311
3,LGBM weighted (no reg),0.9568,0.8371,0.8573,11.9012,8.1284,0.1197
4,"Ensemble (weighted, no reg)",0.9620,0.8411,0.8585,11.7886,8.0488,0.1209
5,XGB auto-tuned (weighted + regularized),0.8971,0.8417,0.8497,12.8897,8.8604,0.0554
6,LGBM auto-tuned (weighted + regularized),0.9125,0.8386,0.8422,12.6276,8.6250,0.0739
7,Final Ensemble (auto-tuned),0.9035,0.8427,0.8502,12.7135,8.6652,0.0608


In [56]:
# LGBM Baseline price band (test set)
lgb_base_band_test = price_band_analysis(
    lgb_tuned, X_test_no_outliers, y_test_no_outliers, "LGBM Baseline (Test)"
)

# Ensemble (weighted, no reg) price band (test set)
ensemble_band_test = price_band_analysis_ensemble(
    ens_test_pred, y_test_no_outliers, "Ensemble weighted no-reg (Test)"
)


=== LGBM Baseline (Test) — Performance by Price Band ===
price_band  n_listings         rmse           mae      mape     mdape
    <$500K        1785 7.799584e+04  51434.454122 15.797741  8.624509
 $500K-$1M        5316 9.739229e+04  66699.302289  8.882893  6.266535
   $1M-$2M        3908 2.246997e+05 162158.488079 11.404992  8.854457
      $2M+        1785 1.477048e+06 695144.956917 16.259711 13.119755
=== Ensemble weighted no-reg (Test) — Performance by Price Band ===
price_band  n_listings         rmse           mae      mape     mdape
    <$500K        1785 7.839699e+04  51247.782697 15.725567  8.614826
 $500K-$1M        5316 1.025071e+05  67950.420677  9.025757  6.296833
   $1M-$2M        3908 2.469308e+05 175914.861349 12.378271  9.564646
      $2M+        1785 1.412712e+06 645936.969161 14.788968 11.304604
